In [10]:
pip install torch torchvision torchaudio timm pandas numpy scikit-learn tqdm pillow


Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

# DEVICE SELECTION (M1 PRO GPU)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU (MPS) 🚀")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU ⚠️ VERY SLOW")


Using Apple Silicon GPU (MPS) 🚀


In [12]:
DATA_ROOT = r"/Users/vinaykumar/Documents/Documents/Vinay Data/NYU Masters/cv proj/kaggle_dataset"

TRAIN_IMG_DIR = os.path.join(DATA_ROOT, 'train_images')
TEST_IMG_DIR  = os.path.join(DATA_ROOT, 'test_images')
TRAIN_CSV     = os.path.join(DATA_ROOT, 'train_ground_truth.csv')
SAMPLE_SUB    = os.path.join(DATA_ROOT, 'sample_submission.csv')
STATE_MAP     = os.path.join(DATA_ROOT, 'state_mapping.csv')

print("TRAIN CSV:", TRAIN_CSV)
print("TRAIN IMAGES:", TRAIN_IMG_DIR)


TRAIN CSV: /Users/vinaykumar/Documents/Documents/Vinay Data/NYU Masters/cv proj/kaggle_dataset/train_ground_truth.csv
TRAIN IMAGES: /Users/vinaykumar/Documents/Documents/Vinay Data/NYU Masters/cv proj/kaggle_dataset/train_images


In [13]:
train_df = pd.read_csv(TRAIN_CSV)
state_map = pd.read_csv(STATE_MAP)

unique_states = sorted(train_df['state_idx'].unique())
state_to_model = {s:i for i,s in enumerate(unique_states)}
model_to_state = {i:s for s,i in state_to_model.items()}

train_df['model_state'] = train_df['state_idx'].map(state_to_model)
NUM_STATES = len(unique_states)

print("States:", NUM_STATES)


States: 33


In [14]:
centroids = train_df.groupby('model_state')[['latitude','longitude']].mean()
centroids_dict = {int(s):centroids.loc[s].values for s in centroids.index}

print("Computed", len(centroids_dict), "state centroids")


Computed 33 state centroids


In [15]:
IMG_SIZE = 224  # smaller = faster on M1 Pro

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2,0.2,0.2,0.1),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class StreetDataset(Dataset):
    def __init__(self, df, root, tf, mode="train"):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.tf = tf
        self.mode = mode

    def load_image(self, fname):
        img = Image.open(os.path.join(self.root, fname)).convert("RGB")
        return self.tf(img)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]

        imgs = torch.stack([
            self.load_image(r['image_north']),
            self.load_image(r['image_east']),
            self.load_image(r['image_south']),
            self.load_image(r['image_west']),
        ])

        out = {'imgs': imgs, 'sample_id': r['sample_id']}

        if self.mode != "test":
            st = int(r['model_state'])
            lat, lon = r['latitude'], r['longitude']
            c_lat, c_lon = centroids_dict[st]
            out['state'] = st
            out['gps_delta'] = torch.tensor([lat - c_lat, lon - c_lon], dtype=torch.float32)

        return out

    def __len__(self):
        return len(self.df)


In [16]:
from sklearn.model_selection import train_test_split

train_df_, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["model_state"]
)

train_ds = StreetDataset(train_df_, TRAIN_IMG_DIR, train_tf, "train")
val_ds   = StreetDataset(val_df, TRAIN_IMG_DIR, val_tf, "val")

# M1 FIX — num_workers MUST be 0
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0)

print("Train:", len(train_ds), "Val:", len(val_ds))


Train: 59382 Val: 6598


In [17]:
def haversine_delta(pred, true):
    return torch.linalg.vector_norm(pred - true, dim=1)

class GeoNet(nn.Module):
    def __init__(self, num_states):
        super().__init__()
        self.backbone = timm.create_model("mobilenetv3_large_100", pretrained=True, num_classes=0)

        # Probe backbone output size
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
        with torch.no_grad():
            out = self.backbone(dummy)
        f = out.shape[1]  # actual number of backbone output features
        print("Backbone feature dim:", f)

        self.fuse = nn.Sequential(
            nn.Linear(f * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.cls = nn.Linear(512, num_states)
        self.reg = nn.Linear(512, 2)

    def forward(self, x):
        B = x.size(0)
        x = x.reshape(B*4, 3, IMG_SIZE, IMG_SIZE)
        feats = self.backbone(x).reshape(B, -1)
        h = self.fuse(feats)
        return self.cls(h), self.reg(h)

model = GeoNet(NUM_STATES).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
ce = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("mps")


Backbone feature dim: 1280


In [18]:
def train_epoch():
    model.train()
    for batch in tqdm(train_loader):
        imgs = batch['imgs'].to(device)
        states = batch['state'].to(device)
        gps = batch['gps_delta'].to(device)

        opt.zero_grad()

        with torch.amp.autocast("mps"):
            logits, pred = model(imgs)
            loss = ce(logits, states) + 0.01 * haversine_delta(pred, gps).mean()

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

def validate():
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch['imgs'].to(device)
            states = batch['state'].to(device)
            logits,_ = model(imgs)
            correct += (logits.argmax(1)==states).sum().item()
    return correct / len(val_ds)

for e in range(10):  # 3 epochs = ~45 min total on M1 Pro
    train_epoch()
    acc = validate()
    print(f"Epoch {e+1} | Val Acc: {acc:.4f}")

torch.save(model.state_dict(), "geonet_mobilenetv3.1.pth")
print("MODEL SAVED ✔")


100%|██████████| 7423/7423 [1:25:08<00:00,  1.45it/s]  


Epoch 1 | Val Acc: 0.5461


100%|██████████| 7423/7423 [1:28:37<00:00,  1.40it/s]   


Epoch 2 | Val Acc: 0.6267


100%|██████████| 7423/7423 [3:40:30<00:00,  1.78s/it]     


...

Epoch 8 | Val Acc: 0.6857


100%|██████████| 7423/7423 [1:05:56<00:00,  1.88it/s]


Epoch 9 | Val Acc: 0.6870


100%|██████████| 7423/7423 [3:17:11<00:00,  1.59s/it]      


Epoch 10 | Val Acc: 0.6808
MODEL SAVED ✔


In [19]:
test_df = pd.read_csv(SAMPLE_SUB)
test_ds = StreetDataset(test_df, TEST_IMG_DIR, val_tf, "test")
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)

model.load_state_dict(torch.load("geonet_mobilenetv3.pth", map_location=device))
model.eval()

rows = []
soft = nn.Softmax(dim=1)

with torch.no_grad():
    for batch in tqdm(test_loader):
        imgs = batch['imgs'].to(device)
        ids = batch['sample_id'].cpu().numpy()

        logits, delta = model(imgs)
        probs = soft(logits).cpu().numpy()
        delta = delta.cpu().numpy()

        for i, sid in enumerate(ids):
            order = probs[i].argsort()[::-1][:5]
            s1 = order[0]

            lat = centroids_dict[s1][0] + delta[i][0]
            lon = centroids_dict[s1][1] + delta[i][1]

            rows.append({
                "sample_id": sid,
                "predicted_state_idx_1": model_to_state[order[0]],
                "predicted_state_idx_2": model_to_state[order[1]],
                "predicted_state_idx_3": model_to_state[order[2]],
                "predicted_state_idx_4": model_to_state[order[3]],
                "predicted_state_idx_5": model_to_state[order[4]],
                "predicted_latitude": lat,
                "predicted_longitude": lon
            })

sub = pd.DataFrame(rows).sort_values("sample_id")
sub.to_csv("submission1.csv", index=False)
print("submission.csv CREATED 🚀")


100%|██████████| 2062/2062 [03:29<00:00,  9.86it/s]

submission.csv CREATED 🚀
